# AR Rehabilitation RL Controller Simulation
## Experiment 2 — Comparative Evaluation of Adaptive Difficulty Policies

### Abstract & Research Objectives
Unilateral Spatial Neglect (USN) stroke rehabilitation requires adaptive target placement to continuously challenge the patient's neglected visual field without causing frustration, $30\text{s}$ timeouts, or cognitive fatigue.

In **Experiment 2**, we evaluate whether **PPO** provides superior adaptive difficulty control compared with alternative Deep RL algorithms and non-learning baselines across **5 independent random seeds**:

| Method | Type | Adaptive? |
| :--- | :--- | :---: |
| **PPO (Recommended)** | Continuous Deep Reinforcement Learning | ✓ |
| **A2C** | Synchronous Actor-Critic RL | ✓ |
| **DQN** | Value-based Deep Q-Network (Discretized Actions) | ✓ |
| **Rule-based** | Heuristic Step Controller | ✓ |
| **Random** | Unadapted Uniform Choice | ❌ |
| **Fixed** | Static Moderate Difficulty | ❌ |

### Rigorous Primary Metrics Evaluated:
1. **Cumulative Reward ($R_{total}$)**: Mean $\pm$ SD accumulated session reward across 5 independent seeds.
2. **Task Success Rate ($SR \%$)**: Percentage of successful target hits before timeout.
3. **Timeout Failure Rate ($TR \%$)**: Percentage of trials resulting in $30\text{s}$ timeouts.
4. **Maximum Neglected-Hemifield Eccentricity ($E_{max}$)**: Maximum target angle reached into the neglected hemifield ($5^\circ - 35^\circ$).
5. **Difficulty Progression ($\Delta E$)**: Total expanded scanning angle into neglected hemifield ($\Delta E = E_{final} - E_{initial}$).
6. **Adaptation Smoothness ($AS$)**: Mathematical metric measuring trial-to-trial adaptation smoothness ($1.0$ = perfectly smooth, lower = erratic staircasing):
   $$AS = 1 - \frac{1}{T-1} \sum_{t=2}^{T} \frac{|e_t - e_{t-1}|}{e_{max} - e_{min}}$$

In [ ]:
# Setup & Package Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, DQN, A2C

from unity_schema import UnitySession, UnityTrial, DifficultyAtTrial
from unity_env import UnityARRehabEnv
from unity_dataset import session_to_observation, predict_next_difficulty
from utils import set_seeds, calculate_multiseed_metrics, BenchmarkPolicies, RuleBasedPolicy
from visualization import plot_experiment2_benchmark, plot_difficulty_progression_comparison, plot_learning_curves

set_seeds(42)
print("Environment & Multi-Seed Benchmark Framework Initialized!")

### 1. Phase 1 — Pre-training Baseline RL Models (PPO, A2C, DQN)
We train PPO, A2C, and DQN models on `UnityARRehabEnv` to establish baseline policies.

In [ ]:
from train_unity import train_unity_agent

rl_results = {}
for algo in ["PPO", "A2C", "DQN"]:
    model, df_logs = train_unity_agent(algo_name=algo, total_timesteps=30000, seed=42)
    rl_results[algo] = df_logs

print("Phase 1 RL Pre-training Complete!")

### 2. Experiment 2 — Multi-Seed Benchmark Evaluation Execution
Evaluates PPO, A2C, DQN, Rule-based, Random, and Fixed methods across **5 independent random seeds**.

In [ ]:
from evaluate import run_experiment2_validation

summary_df, progression_dict = run_experiment2_validation(seeds=[42, 101, 202, 303, 404], num_episodes=20)
display(summary_df[['Method', 'Adaptive', 'reward_str', 'success_rate_pct', 'timeout_rate_pct', 'max_eccentricity_deg', 'delta_eccentricity_deg', 'smoothness_str']])

### 3. Publication Figures & Research Visualizations

In [ ]:
from IPython.display import Image, display
from config import OUTPUT_DIR

exp2_bar_img = os.path.join(OUTPUT_DIR, "exp2_benchmark_comparison.png")
exp2_traj_img = os.path.join(OUTPUT_DIR, "exp2_difficulty_progression.png")
rl_curves_img = os.path.join(OUTPUT_DIR, "learning_curves_comparison.png")

print("Figure 1: Experiment 2 — Multi-Seed Method Comparison Bar Charts (Mean ± SD)")
if os.path.exists(exp2_bar_img):
    display(Image(filename=exp2_bar_img))

print("Figure 2: Trial Difficulty Progression Trajectory (Rule-Based Staircasing vs PPO Smooth Adaptation)")
if os.path.exists(exp2_traj_img):
    display(Image(filename=exp2_traj_img))

print("Figure 3: 3-RL Algorithm Learning Curves Comparison (PPO vs DQN vs A2C)")
if os.path.exists(rl_curves_img):
    display(Image(filename=rl_curves_img))